In [1]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


# 📥 Étape 1 : Acquisition des Données & Multi-Sources

Cette étape correspond au premier chapitre du pipeline de Data Science : identifier, importer et consolider nos jeux de données bruts issus de différentes sources.

**Projet :** prédire le résultat de matchs de football internationaux, avec pour objectif final d'estimer le vainqueur de la Coupe du Monde 2026.

**Sources mobilisées :**

| Source | Fichier | Origine |
|---|---|---|
| Résultats des matchs (principale) | `results.csv` | Kaggle — *International football results from 1872* (martj42) |
| Séances de tirs au but | `shootouts.csv` | Kaggle (martj42) |
| Buteurs match par match | `goalscorers.csv` | Kaggle (martj42) |
| Anciens noms de pays | `former_names.csv` | Kaggle (martj42) |
| Classement FIFA historique (1992 → 2024) | `fifa_ranking-2024-06-20.csv` | Kaggle — *FIFA World Ranking* |

> ⚠️ Le classement FIFA est **historisé** : chaque match reçoit le classement réellement en vigueur **à sa date**, et non un classement actuel. C'est indispensable pour une prédiction fiable — appliquer un classement futur à un match passé serait une *fuite de données*.

### 1. Initialisation de l'environnement

In [2]:
import os
import sys
import pandas as pd
import numpy as np

# Ajout du dossier parent au chemin de recherche des modules
sys.path.append(os.path.abspath('..'))

print("Libraries importées avec succès ! Prêt pour l'étape d'Acquisition des Données.")

Libraries importées avec succès ! Prêt pour l'étape d'Acquisition des Données.


### 2. Chargement de la source de données principale

Notre jeu de données principal est `results.csv` : il recense **un match international par ligne** depuis 1872 (équipes, score, compétition, ville, pays, terrain neutre). C'est la base de notre futur modèle de prédiction.

In [3]:
# Jeu de données principal : résultats des matchs internationaux de football.
main_data_path = '../data/raw/results.csv'

# Chargement brut avec Pandas (le nettoyage interviendra à l'étape 2 - Wrangling).
df_results = pd.read_csv(main_data_path)

print(f"Dimensions du dataset principal : {df_results.shape}")
print(f"Colonnes : {list(df_results.columns)}")
df_results.head()

Dimensions du dataset principal : (49287, 9)
Colonnes : ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


### 3. Intégration de données secondaires (Multi-Sources)

Pour enrichir l'analyse, nous mobilisons **quatre sources complémentaires**. Les trois premières proviennent du même dataset Kaggle que les résultats. La quatrième est l'**historique du classement mondial FIFA** : une ligne = le classement d'une équipe à une date de publication donnée, depuis 1992.

In [4]:
# Chargement des sources secondaires.
df_shootouts    = pd.read_csv('../data/raw/shootouts.csv')      # séances de tirs au but
df_goalscorers  = pd.read_csv('../data/raw/goalscorers.csv')    # buteurs match par match
df_former_names = pd.read_csv('../data/raw/former_names.csv')   # anciens noms de pays

# Historique du classement FIFA : 1 ligne = classement d'une équipe à une date donnée.
df_ranking = pd.read_csv('../data/raw/fifa_ranking-2024-06-20.csv')

# Aperçu des dimensions de chaque source.
for nom, df_src in [('shootouts', df_shootouts), ('goalscorers', df_goalscorers),
                    ('former_names', df_former_names), ('ranking FIFA', df_ranking)]:
    print(f"{nom:14s} : {df_src.shape[0]:>6} lignes, {df_src.shape[1]} colonnes")

df_ranking.head()

shootouts      :    675 lignes, 5 colonnes
goalscorers    :  47601 lignes, 8 colonnes
former_names   :     36 lignes, 4 colonnes
ranking FIFA   :  67472 lignes, 8 colonnes


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,140.0,Brunei Darussalam,BRU,2.0,0.0,140,AFC,1992-12-31
1,33.0,Portugal,POR,38.0,0.0,33,UEFA,1992-12-31
2,32.0,Zambia,ZAM,38.0,0.0,32,CAF,1992-12-31
3,31.0,Greece,GRE,38.0,0.0,31,UEFA,1992-12-31
4,30.0,Algeria,ALG,39.0,0.0,30,CAF,1992-12-31


### 4. Fusion des sources

C'est l'étape clé. Elle enchaîne trois opérations :

1. **Filtrage temporel** — le classement FIFA n'existe que depuis fin 1992. On ne conserve donc que les matchs joués **à partir du 31/12/1992** (les matchs antérieurs n'ont aucun classement disponible).

2. **Jointure temporelle du classement FIFA** — pour chaque match, on récupère le classement **en vigueur à sa date** grâce à `pd.merge_asof`, qui associe la dernière publication du classement *antérieure ou égale* à la date du match (`direction='backward'`). On le fait deux fois : équipe à domicile, puis équipe à l'extérieur. ⚠️ Associer un classement *postérieur* au match serait une **fuite de données** et fausserait la prédiction.

3. **Jointure des tirs au but** — au niveau du match (clé : date + équipes).

Au passage, on **harmonise les noms d'équipes** : quelques pays sont orthographiés différemment dans les deux sources (ex. *USA* vs *United States*, *Czechia* vs *Czech Republic*). On aligne les noms du classement sur ceux de `results.csv` pour que la jointure fonctionne.

Les buteurs (`goalscorers`) et les anciens noms (`former_names`) sont chargés mais pas fusionnés ici : ils seront exploités plus tard (analyse des buts à l'EDA, uniformisation fine des noms au Wrangling).

In [5]:
# --- Préparation : conversion des dates ---
# merge_asof exige des colonnes de type datetime triées.
df_results['date'] = pd.to_datetime(df_results['date'])
df_ranking['rank_date'] = pd.to_datetime(df_ranking['rank_date'])

# --- 1. Filtrage temporel : on ne garde que les matchs depuis le 1er classement FIFA ---
premiere_date = df_ranking['rank_date'].min()
df = df_results[df_results['date'] >= premiere_date].copy()
print(f"Matchs conservés (depuis {premiere_date.date()}) : {len(df)} / {len(df_results)}")

# --- 2. Harmonisation des noms d'équipes (clés de jointure) ---
# Certains pays sont nommés différemment dans le classement et dans results.csv.
# On aligne les noms du classement sur ceux de results.csv.
noms_fifa_vers_results = {
    'USA': 'United States',          'Korea Republic': 'South Korea',
    'Korea DPR': 'North Korea',       'IR Iran': 'Iran',
    "Côte d'Ivoire": 'Ivory Coast',   'Czechia': 'Czech Republic',
    'Congo DR': 'DR Congo',           'Cabo Verde': 'Cape Verde',
    'Curacao': 'Curaçao',             'Brunei Darussalam': 'Brunei',
    'Chinese Taipei': 'Taiwan',       'Kyrgyz Republic': 'Kyrgyzstan',
    'The Gambia': 'Gambia',
    'St Kitts and Nevis': 'Saint Kitts and Nevis',
    'St Lucia': 'Saint Lucia',
    'St Vincent and the Grenadines': 'Saint Vincent and the Grenadines',
}
df_ranking['country_full'] = df_ranking['country_full'].replace(noms_fifa_vers_results)

# --- 3. Jointure temporelle du classement FIFA (le bon classement à la date du match) ---
df = df.sort_values('date')
rk = df_ranking[['rank_date', 'country_full', 'rank', 'total_points']].sort_values('rank_date')

# Classement de l'équipe à domicile, valide à la date du match.
df = pd.merge_asof(
    df,
    rk.rename(columns={'country_full': 'home_team',
                       'rank': 'home_rank', 'total_points': 'home_points'}),
    left_on='date', right_on='rank_date', by='home_team', direction='backward',
).drop(columns='rank_date')

# Classement de l'équipe à l'extérieur, valide à la date du match.
df = pd.merge_asof(
    df,
    rk.rename(columns={'country_full': 'away_team',
                       'rank': 'away_rank', 'total_points': 'away_points'}),
    left_on='date', right_on='rank_date', by='away_team', direction='backward',
).drop(columns='rank_date')

# --- 4. Jointure des tirs au but (au niveau du match) ---
shootouts = df_shootouts.rename(columns={'winner': 'shootout_winner'}).copy()
shootouts['date'] = pd.to_datetime(shootouts['date'])
df_merged = df.merge(
    shootouts[['date', 'home_team', 'away_team', 'shootout_winner']],
    on=['date', 'home_team', 'away_team'], how='left',
)

print(f"Dimensions après fusion : {df_merged.shape}")
df_merged.head()

Matchs conservés (depuis 1992-12-31) : 30583 / 49287
Dimensions après fusion : (30583, 14)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_rank,home_points,away_rank,away_points,shootout_winner
0,1993-01-01,Ghana,Mali,1.0,1.0,Friendly,Libreville,Gabon,True,39.0,34.0,69.0,22.0,NaN
1,1993-01-02,Gabon,Burkina Faso,1.0,1.0,Friendly,Libreville,Gabon,False,55.0,27.0,97.0,11.0,NaN
2,1993-01-02,Kuwait,Lebanon,2.0,0.0,Friendly,Kuwait City,Kuwait,False,71.0,21.0,161.0,0.0,NaN
3,1993-01-03,Burkina Faso,Mali,1.0,0.0,Friendly,Libreville,Gabon,True,97.0,11.0,69.0,22.0,NaN
4,1993-01-03,Gabon,Ghana,2.0,3.0,Friendly,Libreville,Gabon,False,55.0,27.0,39.0,34.0,NaN


### 5. Consignation des données consolidées

Nous sauvegardons le tableau consolidé dans `data/processed/`. Il servira de point d'entrée à l'étape suivante (**02 — Data Wrangling**), qui se chargera de le nettoyer.

In [6]:
# Le dossier data/processed/ accueille les fichiers générés par le pipeline.
os.makedirs('../data/processed', exist_ok=True)

output_path = '../data/processed/matches_merged.csv'
df_merged.to_csv(output_path, index=False)

print(f"Acquisition terminée : {df_merged.shape[0]} matchs consolidés.")
print(f"Fichier écrit : {output_path}")

Acquisition terminée : 30583 matchs consolidés.
Fichier écrit : ../data/processed/matches_merged.csv
